# Module 4 — Evaluation & Analysis


## Dataset selection

Set `DATASET` in Cell 1 to choose which extraction to evaluate:
- `"test"` → `atlas_test_5x5x10_validated_prompts.h5`
- `"small"` → `atlas_small_40x50x50_validated_prompts.h5`

In [ ]:
# Cell 1 – Setup & load atlas
import subprocess, sys, os, shutil
for pkg in ["h5py", "seaborn", "matplotlib", "numpy", "pandas", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── Detect environment ───────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount Drive (force-clean stale mounts)
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

# ── Dataset selector ─────────────────────────────────────────────────────
# Change this to switch between datasets:
DATASET = "small"    # "test" or "small"

_DATASETS = {
    "test":  "atlas_test_5x5x10_validated_prompts",
    "small": "atlas_small_40x50x50_validated_prompts",
}
_base = _DATASETS[DATASET]

if IN_COLAB:
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

ATLAS_HDF5 = f"{DATA_DIR}/{_base}.h5"
STATS_JSON = f"{DATA_DIR}/{_base}_stats.json"

# ── Imports ──────────────────────────────────────────────────────────────
LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, h5py, json
from module2.io_utils import load_atlas_hdf5

# ── Load atlas ───────────────────────────────────────────────────────────
print(f"Environment      : {'Colab' if IN_COLAB else 'Local'}")
print(f"Loading: {ATLAS_HDF5}")
atlas = load_atlas_hdf5(ATLAS_HDF5)
pair_masks       = atlas["pair_masks"]
universal_masks  = atlas["universal_masks"]
metrics          = atlas["metrics"]
metadata         = atlas["metadata"]

print(f"Dataset          : {DATASET}")
print(f"  Pairs          : {len(pair_masks)}")
print(f"  Universal AST  : {len(universal_masks['ast'])}")
print(f"  Universal Blt  : {len(universal_masks['builtin'])}")
print(f"  Metadata       : {metadata}")

## Global Density Heatmap

Concatenate all universal circuit masks (AST + Builtin) across all 8 layers into a single binary matrix. Each row is a circuit, each column is a neuron (layer 0–7, 2048 neurons per layer = 16,384 total). Dark = active neuron.

In [ ]:
# Cell 2 – Global Density Heatmap
import numpy as np, matplotlib.pyplot as plt, seaborn as sns

layer_ids = sorted({lid for lm in pair_masks.values() for lid in lm})

# Collect all universal circuit names and their concatenated masks
circuit_names = []
mask_rows = []

for name in sorted(universal_masks["ast"]):
    vec = []
    for lid in layer_ids:
        m = universal_masks["ast"][name].get(lid)
        if m is not None:
            vec.append(m.astype(np.float32))
    if vec:
        circuit_names.append(f"AST:{name}")
        mask_rows.append(np.concatenate(vec))

for name in sorted(universal_masks["builtin"]):
    vec = []
    for lid in layer_ids:
        m = universal_masks["builtin"][name].get(lid)
        if m is not None:
            vec.append(m.astype(np.float32))
    if vec:
        circuit_names.append(f"BLT:{name}")
        mask_rows.append(np.concatenate(vec))

mask_matrix = np.array(mask_rows)
n_circuits, n_neurons = mask_matrix.shape
neurons_per_layer = n_neurons // len(layer_ids)

print(f"Circuits: {n_circuits} | Neurons: {n_neurons} | Layers: {len(layer_ids)}")

# Plot
fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(mask_matrix, cmap="Blues", cbar=False, yticklabels=False, ax=ax)
ax.set_title(f"Global Density Heatmap ({n_circuits} circuits x {n_neurons} neurons)")
ax.set_xlabel(f"Neurons (Layer {layer_ids[0]} to Layer {layer_ids[-1]})")
ax.set_ylabel(f"Circuits ({n_circuits} AST types / builtins)")

# Layer boundary lines
for i in range(1, len(layer_ids)):
    ax.axvline(i * neurons_per_layer, color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Filtered Heatmap — Dark Neurons Removed

Same heatmap but with columns (neurons) that never participate in any circuit removed. This compresses out the "dark" neurons to reveal structure in the active region.

In [ ]:
# Cell 3 – Heatmap without never-firing neurons
col_sums = mask_matrix.sum(axis=0)
active_cols = col_sums > 0
filtered_matrix = mask_matrix[:, active_cols]

n_dark = int((~active_cols).sum())
n_active = int(active_cols.sum())
print(f"Total neurons: {n_neurons} | Active: {n_active} | Dark (never fire): {n_dark}")

fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(filtered_matrix, cmap="Blues", cbar=False, yticklabels=False, ax=ax)
ax.set_title(f"Density Heatmap — dark neurons removed ({n_circuits} circuits x {n_active} active neurons)")
ax.set_xlabel("Active neurons (dark neurons excluded)")
ax.set_ylabel(f"Circuits ({n_circuits} AST types / builtins)")

# Layer boundary lines — recompute positions after filtering
running = 0
for i in range(len(layer_ids)):
    layer_slice = active_cols[i * neurons_per_layer : (i + 1) * neurons_per_layer]
    running += int(layer_slice.sum())
    if i < len(layer_ids) - 1:
        ax.axvline(running, color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Filtered Heatmap — Dark + Always-On Neurons Removed

Removes both extremes: neurons that never fire in any circuit (dark) and neurons that fire in every circuit (always-on / non-selective). What remains are the **selective** neurons — the ones that distinguish circuits from each other. Per-layer breakdown reported below.

In [ ]:
# Cell 4 – Heatmap without dark + always-on neurons, per-layer report
always_on = col_sums == n_circuits
selective_cols = active_cols & ~always_on
selective_matrix = mask_matrix[:, selective_cols]

n_always = int(always_on.sum())
n_selective = int(selective_cols.sum())

# ── Per-layer breakdown ──────────────────────────────────────────────────
print(f"{'Layer':>5} | {'Total':>5} | {'Dark':>5} | {'Always-On':>9} | {'Selective':>9}")
print("-" * 50)
for i, lid in enumerate(layer_ids):
    s = i * neurons_per_layer
    e = s + neurons_per_layer
    layer_col_sums = col_sums[s:e]
    l_total = neurons_per_layer
    l_dark = int((layer_col_sums == 0).sum())
    l_always = int((layer_col_sums == n_circuits).sum())
    l_selective = l_total - l_dark - l_always
    print(f"{lid:>5} | {l_total:>5} | {l_dark:>5} | {l_always:>9} | {l_selective:>9}")

print("-" * 50)
print(f"{'ALL':>5} | {n_neurons:>5} | {n_dark:>5} | {n_always:>9} | {n_selective:>9}")

# ── Plot ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(selective_matrix, cmap="Blues", cbar=False, yticklabels=False, ax=ax)
ax.set_title(f"Density Heatmap — selective neurons only ({n_circuits} circuits x {n_selective} neurons)")
ax.set_xlabel("Selective neurons (dark + always-on excluded)")
ax.set_ylabel(f"Circuits ({n_circuits} AST types / builtins)")

# Layer boundary lines
running = 0
for i in range(len(layer_ids)):
    layer_slice = selective_cols[i * neurons_per_layer : (i + 1) * neurons_per_layer]
    running += int(layer_slice.sum())
    if i < len(layer_ids) - 1:
        ax.axvline(running, color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Circuit × Selective Neuron Table

DataFrame where rows = circuits (AST + Builtin), columns = selective neurons. Each entry = number of active selective neurons for that circuit. Displayed as a styled heatmap table.

In [ ]:
# Cell 5 – Circuit x Selective Neurons table (active count per layer)
import pandas as pd

# Build a table: rows = circuit names, columns = layers, values = # active selective neurons
selective_idx = np.where(selective_cols)[0]

rows = []
for ci, cname in enumerate(circuit_names):
    row = {"circuit": cname}
    for i, lid in enumerate(layer_ids):
        s = i * neurons_per_layer
        e = s + neurons_per_layer
        # Selective neurons in this layer
        layer_sel = selective_idx[(selective_idx >= s) & (selective_idx < e)]
        row[f"L{lid}"] = int(mask_matrix[ci, layer_sel].sum())
    row["total"] = int(selective_matrix[ci].sum())
    rows.append(row)

table_df = pd.DataFrame(rows).set_index("circuit")
table_df = table_df.sort_values("total", ascending=False)

print(f"Circuits: {len(table_df)} | Selective neurons: {n_selective}")
print()
table_df

In [ ]:
# Cell 6 – Layer 5 deep dive: heatmap + column/row sum tables
FOCUS_LAYER = 5
layer_idx = layer_ids.index(FOCUS_LAYER)
s = layer_idx * neurons_per_layer
e = s + neurons_per_layer

# Extract selective neurons for this layer only
layer_selective = selective_cols[s:e]
layer_sel_indices = np.where(layer_selective)[0]  # local indices within layer
n_sel_layer = len(layer_sel_indices)

# Build matrix: circuits x selective neurons at this layer
layer_matrix = mask_matrix[:, s:e][:, layer_selective].astype(np.float32)

# ── Heatmap ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(12, n_sel_layer * 0.15), max(8, n_circuits * 0.18)))
sns.heatmap(layer_matrix, cmap="Blues", cbar=False,
            xticklabels=layer_sel_indices, yticklabels=circuit_names, ax=ax)
ax.set_title(f"Layer {FOCUS_LAYER} — activation map ({n_circuits} circuits x {n_sel_layer} selective neurons)")
ax.set_xlabel(f"Selective neuron index (within layer {FOCUS_LAYER})")
ax.set_ylabel("Circuit")
ax.tick_params(axis="x", labelsize=6, rotation=90)
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

# ── Table 1: column sums (per neuron — how many circuits use it) ─────────
neuron_sums = layer_matrix.sum(axis=0).astype(int)
col_df = pd.DataFrame({
    "neuron": layer_sel_indices,
    "n_circuits": neuron_sums,
}).sort_values("n_circuits", ascending=False).reset_index(drop=True)

print(f"\n{'='*50}")
print(f"Layer {FOCUS_LAYER} — Per-neuron activation count (column sums)")
print(f"Selective neurons: {n_sel_layer} | "
      f"Mean: {neuron_sums.mean():.1f} | "
      f"Min: {neuron_sums.min()} | Max: {neuron_sums.max()}")
print(f"{'='*50}")
display(col_df)

# ── Table 2: row sums (per circuit — how many selective neurons it uses) ─
circuit_sums = layer_matrix.sum(axis=1).astype(int)
row_df = pd.DataFrame({
    "circuit": circuit_names,
    "n_selective_neurons": circuit_sums,
}).sort_values("n_selective_neurons", ascending=False).reset_index(drop=True)

print(f"\n{'='*50}")
print(f"Layer {FOCUS_LAYER} — Per-circuit activation count (row sums)")
print(f"Circuits: {n_circuits} | "
      f"Mean: {circuit_sums.mean():.1f} | "
      f"Min: {circuit_sums.min()} | Max: {circuit_sums.max()}")
print(f"{'='*50}")
display(row_df)

In [ ]:
# Cell 7 – Proprietary neurons per circuit at Layer 5
# A "proprietary" neuron fires in exactly one circuit and no other.
proprietary_mask = (neuron_sums == 1)
n_proprietary = int(proprietary_mask.sum())

# For each proprietary neuron, find which circuit owns it
prop_matrix = layer_matrix[:, proprietary_mask]
prop_per_circuit = prop_matrix.sum(axis=1).astype(int)

# Build table — only circuits with >= 1 proprietary neuron
prop_rows = []
for ci, cname in enumerate(circuit_names):
    if prop_per_circuit[ci] > 0:
        prop_rows.append({"circuit": cname, "proprietary_neurons": prop_per_circuit[ci]})

prop_df = pd.DataFrame(prop_rows).sort_values("proprietary_neurons", ascending=False).reset_index(drop=True)

print(f"Layer {FOCUS_LAYER} — Proprietary neurons (active in exactly 1 circuit)")
print(f"Total proprietary: {n_proprietary} / {n_sel_layer} selective neurons")
print(f"Circuits with >= 1: {len(prop_df)} / {n_circuits}")
print()
display(prop_df)

In [ ]:
# Cell 8 – Double-ownership neurons at Layer 5
# Neurons that fire in exactly 2 circuits — shared by a specific pair.
double_mask = (neuron_sums == 2)
n_double = int(double_mask.sum())

double_matrix = layer_matrix[:, double_mask]
double_per_circuit = double_matrix.sum(axis=1).astype(int)

# Per-circuit table
double_rows = []
for ci, cname in enumerate(circuit_names):
    if double_per_circuit[ci] > 0:
        double_rows.append({"circuit": cname, "shared_neurons": double_per_circuit[ci]})

double_df = pd.DataFrame(double_rows).sort_values("shared_neurons", ascending=False).reset_index(drop=True)

print(f"Layer {FOCUS_LAYER} — Double-ownership neurons (active in exactly 2 circuits)")
print(f"Total double-ownership: {n_double} / {n_sel_layer} selective neurons")
print(f"Circuits involved: {len(double_df)} / {n_circuits}")
print()
display(double_df)

# Which pairs share these neurons?
double_global = np.where(double_mask)[0]
pair_rows = []
for di in double_global:
    owners = [circuit_names[ci] for ci in range(n_circuits) if layer_matrix[ci, di] > 0]
    pair_rows.append({
        "neuron": layer_sel_indices[di],
        "circuit_1": owners[0],
        "circuit_2": owners[1],
    })

pair_df = pd.DataFrame(pair_rows).sort_values(["circuit_1", "circuit_2"]).reset_index(drop=True)
print(f"\nSharing pairs ({len(pair_df)} neurons):")
display(pair_df)

In [ ]:
# Cell 9 – Layer 5: neuron ownership distribution
from collections import Counter

ownership_counts = Counter(neuron_sums)
max_owners = max(ownership_counts.keys())

print(f"Layer {FOCUS_LAYER} — Selective neuron ownership distribution")
print(f"{'Owners':>8} | {'Neurons':>8} | {'%':>6} | {'Bar'}")
print("-" * 50)
for k in range(1, max_owners + 1):
    n = ownership_counts.get(k, 0)
    pct = 100 * n / n_sel_layer
    bar = "#" * int(pct / 2)
    print(f"{k:>8} | {n:>8} | {pct:>5.1f}% | {bar}")

print("-" * 50)
print(f"{'Total':>8} | {n_sel_layer:>8} |")

## Circuit Similarity — Layer 5

Three complementary measures on selective neurons:
- **Jaccard**: overlap relative to combined footprint
- **Overlap Coefficient**: detects subset/containment (is one circuit inside another?)
- **Phi Coefficient (MCC)**: detects correlation, independence, or anti-correlation

In [ ]:
# Cell 10 – Circuit similarity at Layer 5: Jaccard, Overlap Coefficient, Phi
import numpy as np, matplotlib.pyplot as plt, matplotlib.colors as mcolors
import seaborn as sns, pandas as pd

n = len(circuit_names)
jaccard_mat = np.zeros((n, n))
overlap_mat = np.zeros((n, n))
phi_mat     = np.zeros((n, n))

for i in range(n):
    a = layer_matrix[i].astype(bool)
    sa = a.sum()
    for j in range(i, n):
        b = layer_matrix[j].astype(bool)
        sb = b.sum()

        tp = int((a & b).sum())
        fp = int((a & ~b).sum())
        fn = int((~a & b).sum())
        tn = int((~a & ~b).sum())
        union = tp + fp + fn

        # Jaccard
        jac = tp / union if union > 0 else 0.0
        jaccard_mat[i, j] = jaccard_mat[j, i] = jac

        # Overlap Coefficient
        min_size = min(sa, sb)
        oc = tp / min_size if min_size > 0 else 0.0
        overlap_mat[i, j] = overlap_mat[j, i] = oc

        # Phi (MCC)
        denom = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
        phi = (tp*tn - fp*fn) / denom if denom > 0 else 0.0
        phi_mat[i, j] = phi_mat[j, i] = phi

# ── Heatmaps ─────────────────────────────────────────────────────────────
show_labels = circuit_names if n <= 50 else False

# Jaccard
fig, ax = plt.subplots(figsize=(min(20, n*0.22+2), min(20, n*0.22+2)))
sns.heatmap(jaccard_mat, ax=ax, vmin=0, vmax=1, cmap="YlOrRd",
            xticklabels=show_labels, yticklabels=show_labels,
            cbar_kws={"label": "Jaccard"})
ax.set_title(f"Jaccard Similarity — Layer {FOCUS_LAYER}")
ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()

# Overlap Coefficient
fig, ax = plt.subplots(figsize=(min(20, n*0.22+2), min(20, n*0.22+2)))
sns.heatmap(overlap_mat, ax=ax, vmin=0, vmax=1, cmap="YlOrRd",
            xticklabels=show_labels, yticklabels=show_labels,
            cbar_kws={"label": "Overlap Coeff"})
ax.set_title(f"Overlap Coefficient — Layer {FOCUS_LAYER}")
ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()

# Phi — diverging colormap (negative = anti-correlated)
fig, ax = plt.subplots(figsize=(min(20, n*0.22+2), min(20, n*0.22+2)))
sns.heatmap(phi_mat, ax=ax, vmin=-1, vmax=1, cmap="RdBu_r", center=0,
            xticklabels=show_labels, yticklabels=show_labels,
            cbar_kws={"label": "Phi (MCC)"})
ax.set_title(f"Phi Coefficient — Layer {FOCUS_LAYER}")
ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 11 – Most interesting pairs from each measure
triu = np.triu_indices(n, k=1)

def top_pairs(mat, names, k=10, ascending=False):
    vals = mat[triu]
    idx = np.argsort(vals) if ascending else np.argsort(vals)[::-1]
    rows = []
    for rank, ix in enumerate(idx[:k]):
        i, j = triu[0][ix], triu[1][ix]
        rows.append({"circuit_1": names[i], "circuit_2": names[j], "value": vals[ix]})
    return pd.DataFrame(rows)

# Highest Jaccard — most overlapping circuits
print(f"Layer {FOCUS_LAYER} — Top 10 most similar (Jaccard)")
display(top_pairs(jaccard_mat, circuit_names, k=10))

# Highest Overlap Coefficient — subset relationships
print(f"\nLayer {FOCUS_LAYER} — Top 10 containment pairs (Overlap Coefficient)")
display(top_pairs(overlap_mat, circuit_names, k=10))

# Highest Phi — most positively correlated
print(f"\nLayer {FOCUS_LAYER} — Top 10 most correlated (Phi > 0)")
display(top_pairs(phi_mat, circuit_names, k=10))

# Most negative Phi — most anti-correlated
print(f"\nLayer {FOCUS_LAYER} — Top 10 most anti-correlated (Phi < 0)")
display(top_pairs(phi_mat, circuit_names, k=10, ascending=True))

# Summary stats
jac_vals = jaccard_mat[triu]
oc_vals  = overlap_mat[triu]
phi_vals = phi_mat[triu]
print(f"\n{'='*60}")
print(f"Layer {FOCUS_LAYER} — Summary ({len(jac_vals)} pairs)")
print(f"  Jaccard  : mean={jac_vals.mean():.4f}  std={jac_vals.std():.4f}")
print(f"  Overlap  : mean={oc_vals.mean():.4f}  std={oc_vals.std():.4f}")
print(f"  Phi      : mean={phi_vals.mean():.4f}  std={phi_vals.std():.4f}")
print(f"  Phi < 0  : {(phi_vals < 0).sum()} pairs ({100*(phi_vals < 0).mean():.1f}%)")
print(f"  Phi > 0.5: {(phi_vals > 0.5).sum()} pairs")
print(f"{'='*60}")